[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Files, Paths and Formats](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)

# Compression and Archives


## What you will be able to do

Look inside a `.zip` and read a file from it without unpacking anything to disk, write archives
of your own, and check an archive from someone else before extracting it.


## The idea

### The problem

Data arrives compressed. A year of daily exports is one `.zip`. A log is `.csv.gz`. A dataset
someone published is a `.tar.gz` with a folder inside it.

The obvious approach is to unpack everything and carry on. That works and has three costs.

**Space.** A 200 MB archive of text can unpack to two gigabytes, and you needed one file from
it.

**Mess.** The unpacked copy sits beside the original, and a week later nobody knows which is
current.

**Trust.** An archive decides its own filenames. Extracting one you did not create means letting
it write wherever those names point, and there is a well-known attack that relies on exactly
that.

You can read a file out of an archive directly, and it is barely more code than opening it.

### The two shapes

> A **zip** or **tar** archive holds **many files with their names**, each compressed
> individually in the zip case. `zipfile` and `tarfile` read and write them.
>
> **gzip** compresses **one stream** and holds no names at all. `readings.csv.gz` is a
> compressed `readings.csv` by convention only; the name lives in the filename, not the file.

That difference decides which you meet where. `.zip` for a bundle sent to a person, `.csv.gz`
for a single large file in a pipeline, `.tar.gz` for a directory tree on Unix.

### What compression does not do

Compression finds repetition. Text repeats heavily, so it shrinks a great deal. Data that is
already compressed, such as a `.jpg`, a `.png` or another zip, has no repetition left and does
not shrink at all.

Small files can come out **larger** than they went in, because the format has a fixed overhead
per member. That is demonstrated below, and it surprises people the first time.

### Where you will meet this

Downloads, exports, backups and anything published as a dataset. The **A Small Pipeline**
notebook at the end of this guide reads its input straight out of an archive.

### What this notebook covers

- Writing a zip, and what the compression actually saved
- Listing what is inside without unpacking
- Reading a member as bytes, and reading a CSV from inside an archive as text
- `gzip` for a single file, and why its mode letters matter
- `tarfile`, which is the same shape with different spelling
- Checking member names before extracting, and the attack that check prevents
- Three errors, plus a mode mistake that gives you bytes when you wanted text

### A first look

Nothing to run yet.

```python
import zipfile, io, csv

with zipfile.ZipFile("archive.zip") as z:
    print(z.namelist())

    with z.open("readings.csv") as member:
        text = io.TextIOWrapper(member, encoding="utf-8", newline="")
        for row in csv.DictReader(text):
            print(row)
```

Nothing was written to disk. The CSV was read out of the compressed archive as it was
decompressed.


## Setup

Seven imports and a folder of files to compress.

- `zipfile` reads and writes `.zip` archives, and is most of this notebook
- `gzip` compresses a single stream, for `.csv.gz` files
- `tarfile` reads and writes `.tar` and `.tar.gz`
- `io` wraps a binary member so it can be read as text
- `csv` reads a member, reusing the **CSV** notebook
- `Path` builds paths and reports file sizes
- `shutil` builds an archive in one line, and removes the scratch folder at the end

**Run this cell before the rest of the notebook.**


In [1]:

import zipfile
import gzip
import tarfile
import io
import csv
from pathlib import Path
import shutil

scratch = Path("scratch")
source = scratch / "source"
source.mkdir(parents=True, exist_ok=True)

(source / "readings.csv").write_text("region,value\nnorth,18.5\nsouth,22.1\n", encoding="utf-8")
(source / "notes.txt").write_text("some notes\n", encoding="utf-8")
(source / "repetitive.txt").write_text("repeat " * 2000, encoding="utf-8")

for path in sorted(source.iterdir()):
    print(f"{path.name:<16} {path.stat().st_size:>6} bytes")


notes.txt            11 bytes
readings.csv         35 bytes
repetitive.txt    14000 bytes


## Worked examples

### Writing a zip


In [2]:

archive = scratch / "archive.zip"

with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(source.iterdir()):
        z.write(path, arcname=path.name)

originals = sum(p.stat().st_size for p in source.iterdir())

print("originals:", originals, "bytes")
print("archive:  ", archive.stat().st_size, "bytes")


originals: 14046 bytes
archive:   414 bytes


`compression=zipfile.ZIP_DEFLATED` is worth passing every time. The default is `ZIP_STORED`,
which writes the files into the archive **without compressing them at all**, and produces a zip
slightly larger than the sum of its contents.

`arcname` sets the name stored inside the archive. Without it, the full path you passed is
stored, so an archive built from `scratch/source/notes.txt` unpacks into
`scratch/source/notes.txt` rather than into wherever the reader wanted it.

### Looking inside without unpacking


In [3]:

with zipfile.ZipFile(archive) as z:
    print(z.namelist())


['notes.txt', 'readings.csv', 'repetitive.txt']


In [4]:

with zipfile.ZipFile(archive) as z:
    for info in z.infolist():
        ratio = info.compress_size / info.file_size if info.file_size else 0
        print(f"{info.filename:<16} {info.file_size:>6} -> {info.compress_size:>6}  {ratio:>5.0%}")


notes.txt            11 ->     13   118%
readings.csv         35 ->     35   100%
repetitive.txt    14000 ->     46     0%


Three very different results, and they are the whole story of what compression does.

`repetitive.txt` is the same word two thousand times, so it shrank to almost nothing.
`readings.csv` had little repetition and barely moved. `notes.txt` came out **larger than it went
in**, because every member carries a fixed overhead and eleven bytes is not enough content to
pay for it.

Compressing a folder of small files, or of files that are already compressed, can make the total
bigger. Check rather than assume.


### Reading a member without extracting

`read` gives the bytes.


In [5]:

with zipfile.ZipFile(archive) as z:
    raw = z.read("notes.txt")

print(raw, type(raw).__name__)
print(raw.decode("utf-8"))


b'some notes\n' bytes
some notes



Bytes, not text, for the reason the **Encodings** notebook gave: the archive stores bytes and
does not record an encoding. Decoding is yours to do, and `utf-8` is the right first guess.

`open` gives a file object instead, which matters when the member is large or when something
expects a file rather than a string.


In [6]:

with zipfile.ZipFile(archive) as z:
    with z.open("readings.csv") as member:
        text = io.TextIOWrapper(member, encoding="utf-8", newline="")
        for row in csv.DictReader(text):
            print(dict(row))


{'region': 'north', 'value': '18.5'}
{'region': 'south', 'value': '22.1'}


That is the pattern worth remembering. `z.open` gives a **binary** file object;
`io.TextIOWrapper` turns it into a text one with an encoding you choose, and anything that reads
text can then use it.

`newline=""` is there for the reason the **CSV** notebook gave, and it applies inside an archive
exactly as it does outside one.

Nothing was written to disk at any point.


### Extracting, when you do want the files


In [7]:

out = scratch / "unpacked"

with zipfile.ZipFile(archive) as z:
    z.extractall(out)

print(sorted(p.name for p in out.iterdir()))


['notes.txt', 'readings.csv', 'repetitive.txt']


In [8]:

one = scratch / "just_one"

with zipfile.ZipFile(archive) as z:
    z.extract("notes.txt", one)

print(sorted(p.name for p in one.iterdir()))


['notes.txt']


### Check the names before you extract

`extractall` writes whatever names the archive contains. An archive you did not create chose
those names, and a member called `../../config.json` writes outside the folder you gave.

That is a real attack with a name, and the defense is four lines.


In [9]:

destination = (scratch / "checked").resolve()

with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        target = (destination / name).resolve()
        safe = target.is_relative_to(destination)
        print(f"{name:<16} safe: {safe}")


notes.txt        safe: True
readings.csv     safe: True
repetitive.txt   safe: True


Every name in this archive stays inside the destination, because this archive was built two
cells ago.

The check is `resolve()` from the **Paths** notebook followed by `is_relative_to`. Resolving is
what collapses any `..` in the name; without it the test would pass on exactly the names it is
meant to catch.


In [10]:

escaping = destination / "../../config.json"

print("resolves to:", "/".join(escaping.resolve().parts[-3:]))
print("inside the destination:", escaping.resolve().is_relative_to(destination))


resolves to: notebooks/files-paths-and-formats/config.json
inside the destination: False


`False`, and that is the answer that should stop an extraction.

Run the check before `extractall` on any archive that came from outside your own code. Python
3.12 added a `filter="data"` argument to `tarfile` that refuses such members for you, and the
principle is the same: do not let a file you were sent choose where it lands.


### gzip: one file, no names


In [11]:

gz = scratch / "readings.csv.gz"

with gzip.open(gz, "wt", encoding="utf-8", newline="") as f:
    f.write((source / "readings.csv").read_text(encoding="utf-8"))

print("original:", (source / "readings.csv").stat().st_size, "bytes")
print("gzipped: ", gz.stat().st_size, "bytes")


original: 35 bytes
gzipped:  73 bytes


Larger again, and for the same reason: gzip has a header, and a 35-byte file has nothing to
compress. On a hundred-megabyte log the ratio is entirely different.

Reading it back is `open` with an extra letter in the mode.


In [12]:

with gzip.open(gz, "rt", encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f):
        print(dict(row))


{'region': 'north', 'value': '18.5'}
{'region': 'south', 'value': '22.1'}


`"rt"` is read-text and `"rb"` is read-bytes. **`gzip.open` defaults to binary**, unlike the
built-in `open` which defaults to text. Forgetting the `t` is the last error in this notebook.

A gzip file holds no filenames. `readings.csv.gz` contains a stream that is a CSV only because
whoever made it said so in the filename, which is why `.gz` files are almost always named after
what is inside them.


### tarfile, and one-line archives

`tarfile` is the same shape with different method names, and it is what Unix data usually
arrives in.


In [13]:

tar = scratch / "archive.tar.gz"

with tarfile.open(tar, "w:gz") as t:
    for path in sorted(source.iterdir()):
        t.add(path, arcname=path.name)

with tarfile.open(tar) as t:
    print("members:", t.getnames())
    print("one member:", t.extractfile("notes.txt").read())

print("tar.gz:", tar.stat().st_size, "bytes | zip:", archive.stat().st_size, "bytes")


members: ['notes.txt', 'readings.csv', 'repetitive.txt']
one member: b'some notes\n'
tar.gz: 369 bytes | zip: 414 bytes


`"w:gz"` means write, compressed with gzip. `"w"` alone makes an uncompressed `.tar`, which is
a container and nothing more.

The tar is slightly smaller here because it compresses the whole stream at once rather than each
member separately. That also means you cannot read one member without decompressing everything
before it, which is the trade: zip is random access, tar.gz is smaller.

When you just want a folder archived and do not care about the details:


In [14]:

made = shutil.make_archive(str(scratch / "bundle"), "zip", source)

print(Path(made).name, Path(made).stat().st_size, "bytes")


bundle.zip 414 bytes


One line, and it takes `"zip"`, `"tar"`, `"gztar"` and a few others. Note it wants the base name
**without** the extension and adds the right one itself.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/08-compression-and-archives-solutions.ipynb).

**1.** Create three text files in `scratch/mine`, then zip them with compression into
`scratch/mine.zip`. Print the archive size against the total of the originals.


In [15]:
# your code here


**2.** List the names inside the archive without extracting it.


In [16]:
# your code here


**3.** Print the original and compressed size of each member, and the ratio.


In [17]:
# your code here


**4.** Read one member as text, without extracting, and print its contents.


In [18]:
# your code here


**5.** Write a CSV, gzip it, and read it back with `csv.DictReader` straight from the `.gz`.


In [19]:
# your code here


**6.** Write the safety check: for every member of your archive, print whether extracting it
into `scratch/target` would stay inside that folder.


In [20]:
# your code here


## Common errors

Each cell below is run on purpose so you can see the real message.

### KeyError: no member by that name


In [21]:

with zipfile.ZipFile(archive) as z:
    z.read("summary.csv")


KeyError: "There is no item named 'summary.csv' in the archive"

`There is no item named 'summary.csv' in the archive`. Names inside an archive often include a
folder prefix, so a member you can see when you double-click the zip may be stored as
`data/summary.csv`.

`namelist()` shows exactly what is there, and is the first thing to print when this happens.


### FileNotFoundError: the archive itself is missing


In [22]:

zipfile.ZipFile(scratch / "not_here.zip")


FileNotFoundError: [Errno 2] No such file or directory: 'scratch/not_here.zip'

The same error any missing file gives. Worth distinguishing from the one above: this is the
archive, that was a member inside it.


### BadZipFile: it is not a zip


In [23]:

zipfile.ZipFile(source / "notes.txt")


BadZipFile: File is not a zip file

`File is not a zip file`. This usually means a download failed part way and you have an HTML
error page with a `.zip` name, which is common enough to check for.

`zipfile.is_zipfile` answers without raising:


In [24]:

for candidate in [archive, source / "notes.txt"]:
    print(f"{candidate.name:<16} is a zip: {zipfile.is_zipfile(candidate)}")


archive.zip      is a zip: True
notes.txt        is a zip: False


### The quiet one: gzip gave you bytes


In [25]:

with gzip.open(gz) as f:
    contents = f.read()

print(type(contents).__name__)
print(contents[:20])
print("first line:", contents.split(b"\n")[0])


bytes
b'region,value\nnorth,1'
first line: b'region,value'


No error. `gzip.open` with no mode defaults to `"rb"`, so `contents` is bytes, and every
operation on it needs `b"..."` literals.

It stops being quiet the moment you treat it as text:


In [26]:

with gzip.open(gz) as f:
    for row in csv.DictReader(f):
        print(row)


Error: iterator should return strings, not bytes (the file should be opened in text mode)

`iterator should return strings, not bytes`. The fix is the mode letter:


In [27]:

with gzip.open(gz, "rt", encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f):
        print(dict(row))


{'region': 'north', 'value': '18.5'}
{'region': 'south', 'value': '22.1'}


The built-in `open` defaults to text and `gzip.open` defaults to binary. That inconsistency is
worth remembering, because the failure appears wherever the data is used rather than where it
was opened.


### Cleaning up


In [28]:

shutil.rmtree(scratch)

print("scratch still there:", scratch.exists())


scratch still there: False


## Recap

- `zipfile` and `tarfile` hold many named files; `gzip` compresses one stream and stores no name.
- Pass `compression=zipfile.ZIP_DEFLATED`, because the default stores without compressing.
- `arcname` controls the name inside the archive, and without it the full path is stored.
- `namelist()` and `infolist()` show what is inside without unpacking anything.
- `z.read(name)` gives bytes; `z.open(name)` wrapped in `io.TextIOWrapper` gives text.
- Small files and already-compressed files can come out **larger**.
- Before extracting an archive you did not create, resolve each member against the destination
  and refuse anything that escapes it.
- `gzip.open` defaults to **binary**, unlike the built-in `open`. Pass `"rt"` for text.
- `shutil.make_archive` builds one in a line and adds the extension itself.


## What is next

The **Writing Safely** notebook, which deals with the other half of file handling: making sure
that a program interrupted halfway through leaves the original data intact rather than a
half-written replacement.


---

&#8592; **Previous:** [Directories](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/07-directories.ipynb)  &nbsp;·&nbsp;  [Files, Paths and Formats Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)  &nbsp;·&nbsp;  **Next:** [Writing Safely](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/09-writing-safely.ipynb) &#8594;
